# Decision Tree Classification Practice

**Topic:** Decision Trees &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(2500, 8)`, `y_train` `(2500,)`: the label is a 4×4 checkerboard over features 0 and 1 on [0,4]² with cell boundaries at 1.3, 2.1, and 3.2, plus ~6% label noise. Features 2–7 are pure noise.
- `X_test` `(900, 8)`: predict these. `y_test` is used by the evaluation cell.

## Your task

1. Verify a linear model fails here (optional but instructive).
2. Fit `sklearn.tree.DecisionTreeClassifier`, tuning `max_depth` (and optionally `min_samples_leaf`) with a validation split or cross-validation.
3. Set `predictions` to your test predictions.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

EDGES = np.array([1.3, 2.1, 3.2])       # uneven cell boundaries

def make(n, seed):
    r = np.random.default_rng(seed)
    grid = r.uniform(0, 4, (n, 2))
    cell = np.digitize(grid[:, 0], EDGES) + np.digitize(grid[:, 1], EDGES)
    y = (cell % 2).astype(int)
    flip = r.random(n) < 0.06
    y = np.where(flip, 1 - y, y)
    noise = r.normal(0, 1, (n, 6))
    return np.hstack([grid, noise]), y

X_train, y_train = make(2500, 71)
X_test, y_test = make(900, 72)
print(f"X_train {X_train.shape}, label balance {y_train.mean():.2f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.85

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

A checkerboard is exactly what axis-aligned splits are good at. The only danger is over/under-fitting the depth, so let cross-validation pick `max_depth` and `min_samples_leaf`.

```python
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

search = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    {"max_depth": [4, 6, 8, 10, 12], "min_samples_leaf": [5, 20, 50]},
    cv=5,
)
search.fit(X_train, y_train)
print("best params:", search.best_params_)
predictions = search.best_estimator_.predict(X_test)
```
